In [5]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Trainer, TrainingArguments


sas_code_list = [
    "proc sort data=old; by var1; run;",
    "proc sort data = old; by var1; run;",
    "proc sort data=old out=new; by var2; run;",
    "proc sort data=alpha; by x; run;",
    "proc sort data=delta; by m; run;",
    "proc sort data=sample1; by col1; run;",
    "proc sort    data =    newdata;  by   var1;   run ;",
    "proc sort   data =   tableA   ;   by  colA ; run ;",
    "proc sort data=new_data   out= sorted_new ;   by  var2  var3; run;",
    "proc sort data = df1 out = new_df; by var1 var2; run;",
    "proc sort data=df1; by var3 var4; run;",
    "proc sort data=table1 out=sorted_table; by varA; run ;",
    "proc sort data = df2 out = lib.df_out ; by varA varB ; run;",
    "proc sort data=table2 out = lib.table2_out ; by varX ; run ;",
    "proc sort data = libdata.sample2 out = outlib.sample2_sorted ; by col2; run;",
    "proc sort data = libdata.data2  out = libdata.sorted2 ; by   z; run ;",
    "proc sort data = dataset1 out = sorted_data; by varA varC; run;",
    "proc sort data=source out=dest; by a b c; run;",
    "proc sort data = beta out = gamma; by y z; run;",
    "proc sort data = old out = outlib.new ; by var1 var2 ; run;",
    "proc sort data = ds1 out = mylib.sorted_ds1 ; by varX varY; run;",
    "proc sort data=old ; by var2 ; run;",
    "proc sort data=sampleX  out= out_sampleX ; by   varX  varY ; run ;",
    "proc sort  data = dsX   out =   sorted_dsX;   by  a  b; run ;",
    "proc sort data = dataSetX;   by  field1  field2  field3; run ;",
    "proc sort    data= library.data1   out = library.sorted1;  by   x  y; run ;",
    "proc sort data = old out = outlib.outset ; by var1; where condition > 1 ; run;",
    "proc sort    data = df3   out = mylib.sorted ; by varX ; where ( varA = 'A' and varB < 5 ) or varC > 10 ; run ;",
    "proc sort data=tableA out=tableA_sorted; by var1 var2; keep var1 var3 var5; run;",
    "proc sort data=source out=dest; by col1 col2; drop col3 col4; where col1 <> 'B'; run;",
    "proc sort data = sales out = sorted_sales; by date; where region = 'North' and (revenue > 1000 or profit > 100); run;",
    "proc sort data=inventory; by product; where stock <= 50; drop supplier price; run;",
    "proc sort data = orders out = orders_sorted; by order_date; where status = 'Completed'; run;",
    "proc sort data = customers out = active_customers; by customer_id; where active = 1; keep customer_id name; run;",
    "proc sort data=transactions out=trans_sorted; by trans_date; where amount >= 1000; run;",
    "proc sort data = logs out = error_logs; by timestamp; where level = 'ERROR'; drop message; run;",
    "proc sort data = employees out = high_earners; by salary; where salary > 70000; run;",
    "proc sort data = products out = discounted_products; by discount; where discount >= 10 and discount <= 50; run;",
    "proc sort data = sessions out = active_sessions; by start_time; where duration > 30; run;",
    "proc sort data = feedback out = positive_feedback; by rating; where rating >= 4; run;",
    "proc sort data = shipments out = delayed_shipments; by ship_date; where delay > 0; run;",
    "proc sort data = attendance out = absent_records; by date; where present = 0; run;",
    "proc sort data = finance out = quarterly_reports; by quarter; where revenue > expense; run;",
    "proc sort data = weather out = rainy_days; by date; where condition = 'Rainy'; run;",
    "proc sort data = survey out = valid_responses; by response_id; where response ne ''; run;",
    "proc sort data = ratings out = top_rated; by score; where score >= 4.5; keep user_id score; run;"
]

pandas_code_list = [
    "old = old.sort_values(['var1'])",
    "old = old.sort_values(['var1'])",
    "new = old.sort_values(['var2'])",
    "alpha = alpha.sort_values(['x'])",
    "delta = delta.sort_values(['m'])",
    "sample1 = sample1.sort_values(['col1'])",
    "newdata = newdata.sort_values(['var1'])",
    "tableA = tableA.sort_values(['colA'])",
    "sorted_new = new_data.sort_values(['var2', 'var3'])",
    "new_df = df1.sort_values(['var1', 'var2'])",
    "df1 = df1.sort_values(['var3', 'var4'])",
    "sorted_table = table1.sort_values(['varA'])",
    "lib_df_out = df2.sort_values(['varA', 'varB'])",
    "lib_table2_out = table2.sort_values(['varX'])",
    "outlib_sample2_sorted = libdata.sample2.sort_values(['col2'])",
    "libdata_sorted2 = libdata.data2.sort_values(['z'])",
    "sorted_data = dataset1.sort_values(['varA', 'varC'])",
    "dest = source.sort_values(['a', 'b', 'c'])",
    "gamma = beta.sort_values(['y', 'z'])",
    "outlib_new = old.sort_values(['var1', 'var2'])",
    "mylib_sorted_ds1 = ds1.sort_values(['varX', 'varY'])",
    "old = old.sort_values(['var2'])",
    "out_sampleX = sampleX.sort_values(['varX', 'varY'])",
    "sorted_dsX = dsX.sort_values(['a', 'b'])",
    "dataSetX = dataSetX.sort_values(['field1', 'field2', 'field3'])",
    "library_sorted1 = library.data1.sort_values(['x', 'y'])",
    "outlib_outset = old.sort_values(['var1'])\noutlib_outset = outlib_outset[outlib_outset['condition'] > 1]",
    "mylib_sorted = df3.sort_values(['varX'])\nmylib_sorted = mylib_sorted[((mylib_sorted['varA'] == 'A') & (mylib_sorted['varB'] < 5)) | (mylib_sorted['varC'] > 10)]",
    "tableA_sorted = tableA.sort_values(['var1', 'var2'])\ntableA_sorted = tableA_sorted[['var1', 'var3', 'var5']]",
    "dest = source.sort_values(['col1', 'col2'])\ndest = dest[dest['col1'] != 'B']\ndest = dest.drop(columns=['col3', 'col4'])",
    "sorted_sales = sales.sort_values(['date'])\nsorted_sales = sorted_sales[(sorted_sales['region'] == 'North') & ((sorted_sales['revenue'] > 1000) | (sorted_sales['profit'] > 100))]",
    "inventory = inventory.sort_values(['product'])\ninventory = inventory[inventory['stock'] <= 50]\ninventory = inventory.drop(columns=['supplier', 'price'])",
    "orders_sorted = orders.sort_values(['order_date'])\norders_sorted = orders_sorted[orders_sorted['status'] == 'Completed']",
    "active_customers = customers.sort_values(['customer_id'])\nactive_customers = active_customers[active_customers['active'] == 1]\nactive_customers = active_customers[['customer_id', 'name']]",
    "trans_sorted = transactions.sort_values(['trans_date'])\ntrans_sorted = trans_sorted[trans_sorted['amount'] >= 1000]",
    "error_logs = logs.sort_values(['timestamp'])\nerror_logs = error_logs[error_logs['level'] == 'ERROR']\nerror_logs = error_logs.drop(columns=['message'])",
    "high_earners = employees.sort_values(['salary'])\nhigh_earners = high_earners[high_earners['salary'] > 70000]",
    "discounted_products = products.sort_values(['discount'])\ndiscounted_products = discounted_products[(discounted_products['discount'] >= 10) & (discounted_products['discount'] <= 50)]",
    "active_sessions = sessions.sort_values(['start_time'])\nactive_sessions = active_sessions[active_sessions['duration'] > 30]",
    "positive_feedback = feedback.sort_values(['rating'])\npositive_feedback = positive_feedback[positive_feedback['rating'] >= 4]",
    "delayed_shipments = shipments.sort_values(['ship_date'])\ndelayed_shipments = delayed_shipments[delayed_shipments['delay'] > 0]",
    "absent_records = attendance.sort_values(['date'])\nabsent_records = absent_records[absent_records['present'] == 0]",
    "quarterly_reports = finance.sort_values(['quarter'])\nquarterly_reports = quarterly_reports[quarterly_reports['revenue'] > quarterly_reports['expense']]",
    "rainy_days = weather.sort_values(['date'])\nrainy_days = rainy_days[rainy_days['condition'] == 'Rainy']",
    "valid_responses = survey.sort_values(['response_id'])\nvalid_responses = valid_responses[valid_responses['response'] != '']",
    "top_rated = ratings.sort_values(['score'])\ntop_rated = top_rated[top_rated['score'] >= 4.5]\ntop_rated = top_rated[['user_id', 'score']]"
]

df = pd.DataFrame({
    "sas_code": sas_code_list,
    "pandas_code": pandas_code_list
})

display(df)

print("\nConvert the DataFrame to a Hugging Face Dataset")
dataset = Dataset.from_pandas(df)
print("Dataset features:", dataset.features)

,sas_code,pandas_code
0,proc sort data=old; by var1; run;,old = old.sort_values(['var1'])
1,proc sort data = old; by var1; run;,old = old.sort_values(['var1'])
2,proc sort data=old out=new; by var2; run;,new = old.sort_values(['var2'])
3,proc sort data=alpha; by x; run;,alpha = alpha.sort_values(['x'])
4,proc sort data=delta; by m; run;,delta = delta.sort_values(['m'])
5,proc sort data=sample1; by col1; run;,sample1 = sample1.sort_values(['col1'])
6,proc sort data = newdata; by var1; ...,newdata = newdata.sort_values(['var1'])
7,proc sort data = tableA ; by colA ; r...,tableA = tableA.sort_values(['colA'])
8,proc sort data=new_data out= sorted_new ; ...,"sorted_new = new_data.sort_values(['var2', 'va..."
9,proc sort data = df1 out = new_df; by var1 var...,"new_df = df1.sort_values(['var1', 'var2'])"



Convert the DataFrame to a Hugging Face Dataset
Dataset features: {'sas_code': Value(dtype='string', id=None), 'pandas_code': Value(dtype='string', id=None)}


In [7]:
model_name = "Salesforce/CodeT5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, legacy = False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [8]:
def tokenize_function(example):
    model_inputs = tokenizer(example['sas_code'], max_length=128, truncation=True)
    labels = tokenizer(text_target=example['pandas_code'], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply tokenization over the entire dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Tokenized example:", tokenized_dataset[0])


Map:   0%|          | 0/46 [00:00<?, ? examples/s]

Tokenized example: {'sas_code': 'proc sort data=old; by var1; run;', 'pandas_code': "old = old.sort_values(['var1'])", 'input_ids': [1, 9381, 1524, 501, 33, 1673, 31, 635, 569, 21, 31, 1086, 31, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1, 1673, 273, 1592, 18, 3804, 67, 2372, 12, 3292, 1401, 21, 19486, 2]}


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",   
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_steps=10,
    weight_decay=0.01,
    run_name="my_custom_run"  
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,  
    eval_dataset=tokenized_dataset    
)

# Start training
trainer.train()


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

In [ ]:
def rule_based_translation(sas_code):
    parsed = parse_sas_sort(sas_code)
    if parsed:
        data = parsed.get("data")
        out = parsed.get("out") if parsed.get("out") else data
        by = parsed.get("by").strip()
        by_vars = [f'"{var.strip()}"' for var in by.split(',')]
        return f"{out} = {data}.sort_values([{', '.join(by_vars)}])"
    return "Invalid SAS Code"
 
sas_code = "proc sort data = dsfs out=jsdhkjs; by dsd; run;"
print("Rule-based Translation:", rule_based_translation(sas_code))